# TSFM — training on Kaggle

One session = one notebook version (*Save Version → Save & Run All*, accelerator **GPU T4 x2**, internet **on**).
Recent PyTorch wheels no longer support the P100 (sm_60); one T4 is used.

Inputs to attach (*Add Input*):
1. the dataset holding `holdout_v1/` (upload of `data/holdouts`), always;
2. from session 2 on: the **output of the previous version of this notebook** (it holds `<RUN>/last.pt`).

Each session trains for `HOURS`, then writes `last.pt` to `/kaggle/working/<RUN>`, and the next one resumes
exactly (same steps, LR schedule, data stream). Keep `TOTAL_EPISODES` / `BATCH_SIZE` / `DATA_CONFIG`
unchanged across sessions: they define the schedule. Private repo: add a `GITHUB_TOKEN` secret (*Add-ons → Secrets*).

In [ ]:
RUN = "v1_kaggle"
MODEL = "v1"
DATA_CONFIG = "configs/train_v1.yaml"
TOTAL_EPISODES = 20_000_000
BATCH_SIZE = 64
NUM_WORKERS = 4          # Kaggle has 4 CPU cores
HOURS = 11.0             # training time of this session (12 h limit, minus setup and checkpoint)
REPO = "https://github.com/rav-lad/tsfm.git"
BRANCH = "model-v0-v1"
TORCH_INDEX = None       # e.g. "https://download.pytorch.org/whl/cu128" if the driver is too old for the locked CUDA 13 wheel

In [ ]:
import glob, os, pathlib, shutil, subprocess

CODE = pathlib.Path("/tmp/tsfm")          # outside /kaggle/working: the notebook output only holds the run
PROJECT = CODE / "tsfm"
RUN_DIR = pathlib.Path("/kaggle/working") / RUN

url = REPO
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    pass  # public repo
if not CODE.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(CODE)], check=True)
print(subprocess.run(["git", "-C", str(CODE), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

subprocess.run(["pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "sync", "--frozen"], cwd=PROJECT, check=True)  # Python 3.13 + CUDA torch from uv.lock
os.environ["UV_NO_SYNC"] = "1"  # later `uv run` calls keep the environment as set up here
if TORCH_INDEX:
    version = subprocess.run(["uv", "run", "python", "-c", "import torch; print(torch.__version__.split('+')[0])"],
                             cwd=PROJECT, capture_output=True, text=True, check=True).stdout.strip()
    subprocess.run(["uv", "pip", "install", "--reinstall", f"torch=={version}", "--index-url", TORCH_INDEX],
                   cwd=PROJECT, check=True)

In [ ]:
# frozen validation suites
found = glob.glob("/kaggle/input/**/holdout_v1", recursive=True)
assert found, "attach the dataset that contains holdout_v1/"
(PROJECT / "data/holdouts").mkdir(parents=True, exist_ok=True)
link = PROJECT / "data/holdouts/holdout_v1"
if not link.exists():
    link.symlink_to(found[0])

# previous session
RUN_DIR.mkdir(parents=True, exist_ok=True)
previous = sorted(glob.glob(f"/kaggle/input/**/{RUN}/last.pt", recursive=True))
if previous and not (RUN_DIR / "last.pt").exists():
    shutil.copytree(os.path.dirname(previous[0]), RUN_DIR, dirs_exist_ok=True)
    print("resuming from", previous[0])
else:
    print("no previous checkpoint: fresh run")

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!cd {PROJECT} && uv run python -c "import torch; ok = torch.cuda.is_available(); print(torch.__version__, 'cuda', torch.version.cuda, 'available', ok); ok and print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0), 'bf16', torch.cuda.is_bf16_supported(including_emulation=False))"
# available False: set TORCH_INDEX in the first cell (CUDA 12.x wheels) and rerun
!nproc && free -g | head -2

In [ ]:
# does on-the-fly generation keep up? (episodes/s with 2 and 4 workers; compare with the ep/s of training)
!cd {PROJECT} && uv run python scripts/profile_loader.py --config {DATA_CONFIG} --workers 2 4 --batches 40

In [ ]:
!cd {PROJECT} && uv run python -u scripts/train.py --model {MODEL} --data-config {DATA_CONFIG}     --run-dir {RUN_DIR} --total-episodes {TOTAL_EPISODES} --batch-size {BATCH_SIZE}     --num-workers {NUM_WORKERS} --stop-after-hours {HOURS} --resume 2>&1 | tee -a {RUN_DIR}/train.log

In [ ]:
!tail -n 3 {RUN_DIR}/metrics.jsonl && ls -la {RUN_DIR}